# Contrastive Retrieval Comparison

Notebook wrapper for the MLP vs atlas-free CNN contrastive text-to-brain and brain-to-text retrieval comparison. The evaluation code lives in `atlas_free_cnn.evaluation.compare_contrastive_retrieval`; this notebook only configures and calls it.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "neurovlm").exists() and (candidate / "experiments" / "3dcnn" / "atlas_free_cnn").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from the neurovlm repo or update this cell.")

REPO_ROOT = find_repo_root()
THREEDCNN = REPO_ROOT / "experiments" / "3dcnn"
MODEL_COMPARISON_DIR = THREEDCNN / "model_comparison"
for path in [REPO_ROOT / "src", THREEDCNN, MODEL_COMPARISON_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

REPO_ROOT

In [ ]:
import pandas as pd

from atlas_free_cnn.evaluation.compare_contrastive_retrieval import (
    CONTRASTIVE_MODEL_IDS,
    CURVES_FILENAME,
    DATASETS,
    DEFAULT_OUTPUT_DIR,
    EXAMPLES_FILENAME,
    SUMMARY_FILENAME,
    run_comparison,
)

list(DATASETS), list(CONTRASTIVE_MODEL_IDS)

## Configure

Set `RUN_MODE` below: `"quick"` is a fast sanity check (MLP on Nilearn only,
two samples -- still downloads the MLP autoencoder and a SPECTER2 text model
on first run, since Nilearn text is embedded on the fly, see
`load_mlp_nilearn_pairs`); `"full"` is the actual comparison across all
datasets/models (also downloads the CNN Stage 3 checkpoints, unified test
split, and normalized SPECTER2 cache). Everything below -- Run Comparison,
Inspect Outputs, Visualize Results -- uses whichever mode you pick here.

In [ ]:
# RUN_MODE:
#   "quick" - fast sanity check: MLP only, Nilearn only, two samples.
#   "full"  - the actual comparison across all datasets/models.
RUN_MODE = "full"

if RUN_MODE == "quick":
    DATASETS_TO_RUN = ["nilearn"]
    MODELS = ["mlp_neurovlm"]
    LIMIT = 2
elif RUN_MODE == "full":
    DATASETS_TO_RUN = ["pubmed", "nilearn", "neurovault"]
    MODELS = [
        "mlp_neurovlm",
        "cnn_contrastive_mixed",
        "cnn_contrastive_pubmed",
        "cnn_contrastive_nilearn",
        "cnn_contrastive_neurovault",
    ]
    LIMIT = 32
else:
    raise ValueError(f"Unknown RUN_MODE {RUN_MODE!r}; expected 'quick' or 'full'")

DEVICE = "cpu"
BATCH_SIZE = 32

preferred_test_jsonl = REPO_ROOT / "experiments" / "3dcnn" / "atlas_free_cnn" / "cache" / "unified_jsonl" / "splits" / "test.jsonl"
TEST_JSONL = preferred_test_jsonl if preferred_test_jsonl.exists() else None

preferred_text_cache = REPO_ROOT / "experiments" / "3dcnn" / "atlas_free_cnn" / "cache" / "text_embeddings" / "specter2_stage3_stage4_emptycentered_unitnorm.pt"
TEXT_EMBEDDING_CACHE = preferred_text_cache if preferred_text_cache.exists() else None

OUTPUT_DIR = REPO_ROOT / DEFAULT_OUTPUT_DIR

{
    "run_mode": RUN_MODE,
    "datasets": DATASETS_TO_RUN,
    "models": MODELS,
    "limit": LIMIT,
    "device": DEVICE,
    "test_jsonl": str(TEST_JSONL) if TEST_JSONL else None,
    "text_embedding_cache": str(TEXT_EMBEDDING_CACHE) if TEXT_EMBEDDING_CACHE else None,
    "output_dir": str(OUTPUT_DIR),
}

## Run Comparison

In [ ]:
result = run_comparison(
    datasets=DATASETS_TO_RUN,
    models=MODELS,
    limit=LIMIT,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    test_jsonl=TEST_JSONL,
    text_embedding_cache=TEXT_EMBEDDING_CACHE,
    batch_size=BATCH_SIZE,
)

result

## Inspect Outputs

In [ ]:
summary = pd.read_csv(result["summary_path"])
summary

In [ ]:
metric_cols = [
    "dataset",
    "requested_model_id",
    "model_id",
    "status",
    "n_pairs",
    "normalized_k_recall_curve_auc",
    "t2i_normalized_k_recall_curve_auc",
    "i2t_normalized_k_recall_curve_auc",
    "mrr",
    "median_rank",
    "checkpoint_path",
    "skip_reason",
]
available = [col for col in metric_cols if col in summary.columns]
summary[available] if available else summary

## Visualize Results

Same fixed model-family colors as the other two comparison notebooks (MLP =
blue, CNN mixed baseline = green, CNN domain-specialized = the dataset's
color). The coverage table shows `status` per (dataset, model) so a missing
checkpoint is visible instead of silently absent from the bars below.
Bidirectional AUC is the mean of Text-to-Brain (T2B) and Brain-to-Text (B2T)
normalized Recall@K AUC; the two directions are also broken out separately
since a model can be much better at one than the other. MLP now runs on all
three datasets: PubMed and NeuroVault use precomputed main-package
resources, and Nilearn is bridged through the atlas-free CNN's packed
volumes (see `load_mlp_nilearn_pairs` / `mlp_masker_bridge`).


In [ ]:
import matplotlib.pyplot as plt
import plotting_utils as pu

pu.coverage_table(summary)

In [ ]:
fig_auc, axes = plt.subplots(1, 3, figsize=(17, 4.5))
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="normalized_k_recall_curve_auc",
    ax=axes[0], title="Bidirectional retrieval AUC (higher is better)", ylabel="Normalized Recall@K AUC", show_legend=False,
)
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="t2i_normalized_k_recall_curve_auc",
    ax=axes[1], title="Text-to-Brain AUC", ylabel="Normalized Recall@K AUC", show_legend=False,
)
pu.grouped_bar(
    summary, category_col="dataset", series_col="model_id", value_col="i2t_normalized_k_recall_curve_auc",
    ax=axes[2], title="Brain-to-Text AUC", ylabel="Normalized Recall@K AUC",
)
fig_auc.suptitle("Contrastive Retrieval Quality by Dataset and Model", x=0.01, ha="left", fontsize=12)
fig_auc.tight_layout(rect=(0, 0, 0.86, 0.95))

In [ ]:
curves_path = result["curves_path"]
curves = pd.read_csv(curves_path) if curves_path.exists() and curves_path.stat().st_size else pd.DataFrame()
curves.head(20)

In [ ]:
fig_recall_curves = pu.small_multiples_lines(
    curves, panel_col="dataset", x_col="normalized_k", y_col="mean_recall", series_col="model_id",
    suptitle="Mean Recall@K Curves by Dataset", xlabel="Normalized K", ylabel="Mean recall", reference_line=True,
)

In [ ]:
examples_path = result["examples_path"]
examples = pd.read_csv(examples_path) if examples_path.exists() and examples_path.stat().st_size else pd.DataFrame()
examples.head(20)

## Existing Output Loader

Use this cell when you have already run the CLI or a previous notebook cell and only want to inspect the saved CSVs.

In [ ]:
saved_summary = OUTPUT_DIR / SUMMARY_FILENAME
saved_curves = OUTPUT_DIR / CURVES_FILENAME
saved_examples = OUTPUT_DIR / EXAMPLES_FILENAME

loaded_summary = pd.read_csv(saved_summary) if saved_summary.exists() and saved_summary.stat().st_size else pd.DataFrame()
loaded_curves = pd.read_csv(saved_curves) if saved_curves.exists() and saved_curves.stat().st_size else pd.DataFrame()
loaded_examples = pd.read_csv(saved_examples) if saved_examples.exists() and saved_examples.stat().st_size else pd.DataFrame()

loaded_summary.head(20)

## Export For Report

Save every figure and dataframe from this run into one directory (PNG +
CSV, plus a `manifest.json` listing them), so an HTML-report builder can
read a single, predictable location instead of re-running the comparison.

In [ ]:
REPORT_ASSETS_DIR = OUTPUT_DIR / "report_assets" / "contrastive_retrieval"

report_manifest = pu.save_report_assets(
    REPORT_ASSETS_DIR,
    figures={"contrastive_retrieval_auc": fig_auc, "contrastive_retrieval_recall_curves": fig_recall_curves},
    dataframes={
        "contrastive_retrieval_summary": summary,
        "contrastive_retrieval_curves": curves,
        "contrastive_retrieval_examples": examples,
    },
)
report_manifest